# 1. Why testing an AI agent is different

We all know how to test normal code: write a function, write a test with `pytest`, run it.
Let's do exactly that, then try the same thing on an AI agent and see where it breaks.

## Testing normal code

A function that picks the cheapest flight from a list:

In [ ]:
%%writefile pricing.py
def cheapest_flight(flights):
    """Return the flight with the lowest price."""
    return min(flights, key=lambda flight: flight["price"])

And a normal test for it:

In [ ]:
%%writefile test_pricing.py
from pricing import cheapest_flight

FLIGHTS = [
    {"flight_number": "BA112", "price": 640},
    {"flight_number": "VS4", "price": 520},
]


def test_cheapest_flight():
    assert cheapest_flight(FLIGHTS)["flight_number"] == "VS4"

In [ ]:
!pytest test_pricing.py -q

Run it again. And again. It passes every time:

In [ ]:
!pytest test_pricing.py -q

This works because normal code is **deterministic**: the same input always gives the same output.
So one `assert` protects us forever.

## Now an AI assistant answers the same question

Here is a small LLM assistant that knows the same two flights:

In [ ]:
%%writefile travel_bot.py
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

SYSTEM_PROMPT = """You are a travel assistant.
Flights from New York to London: BA112 costs $640, VS4 costs $520."""


def ask_travel_bot(question):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
    )
    return response.choices[0].message.content

We ask it the same question three times:

In [ ]:
from travel_bot import ask_travel_bot

for run in range(3):
    print(f"Run {run + 1}:", ask_travel_bot("Which flight to London is the cheapest?"), "\n")

All three answers are correct. Look at the wording: it can change from run to run, and it almost never matches the exact sentence we would write in a test.

## Try to test it with pytest

The way we test normal code: compare the output with the expected output.

In [ ]:
%%writefile test_travel_bot.py
from travel_bot import ask_travel_bot


def test_exact_answer():
    answer = ask_travel_bot("Which flight to London is the cheapest?")
    assert answer == "The cheapest flight is VS4 for $520."


def test_answer_mentions_vs4():
    answer = ask_travel_bot("Which flight to London is the cheapest?")
    assert "VS4" in answer

In [ ]:
!pytest test_travel_bot.py -q

- `test_exact_answer` **fails**, even though the bot was right. The words simply didn't match.
- `test_answer_mentions_vs4` **passes**. So is a keyword check the answer? Look at this reply:

In [ ]:
wrong_answer = "Please don't book VS4, it is not available. The cheapest option is BA112 for $640."

print("Mentions VS4?", "VS4" in wrong_answer)

The keyword test **passes a wrong answer**. Checking words cannot tell us what an answer **means**.

## It gets worse with agents

An **agent** does not only write text. It calls tools: it searches, it books, it cancels.
The final answer can say *"Done! Booked VS4 for you"* even if, behind the scenes, it:

- searched the wrong route
- booked before checking the price
- booked twice
- cancelled someone else's booking

None of this is visible in the text. A text test cannot see it at all.

## Three problems normal tests cannot solve

| Problem | Example | What we need instead |
|---|---|---|
| The words can change on every run | `answer == "..."` fails on a correct answer | test the **behaviour**, not the exact words |
| Keywords don't capture meaning | "don't book VS4" contains "VS4" | a **judge** that understands meaning |
| The answer hides the steps | "Booked!" after calling the wrong tool | check **what the agent did**, not only what it said |

And one more difference: instead of a strict yes or no, we get a **score from 0 to 1** and decide the **threshold** that counts as good enough.

That is what an evaluation framework gives us. We will use **DeepEval**, and at the end of the class you will see that it still runs inside `pytest`.

**Next:** notebook 2, what agent evaluation is.